# Deep Learning & Transformer Models — Next-15-minute Traffic Forecast

**Task (from the project README):** predict the vehicle count of the next 15 minutes at an intersection from its recent history, then turn the predicted volume into a congestion class (Low / Moderate / High / Severe).

**Models compared** (code in `src/deep_learning/`)

| Model | Idea |
|---|---|
| `persistence` | "next 15 min = right now" — the naive forecast every model must beat |
| `hist_gb_tabular` | gradient boosting on the flattened 4-hour window — strong classical reference |
| `mlp` | feed-forward network on the flattened window |
| `lstm`, `gru` | recurrent networks over the 16 time steps |
| `transformer` | encoder-only self-attention model over the 16 time steps |

**Setup that keeps the comparison honest**
- Input: last 16 steps (4 h) × 23 features; the target is next-step *utilisation* (count ÷ road capacity), converted back to vehicles for the metrics.
- **Chronological split**: first 70 % of the timeline = train, next 15 % = validation, last 15 % = test (June 9 – July 4, 2025). Scaling statistics come from the training period only.
- `target_next_count` / `target_congestion` are **never** used as inputs (the congestion label is computed from the target count, so using either would leak the answer).

> Needs PyTorch: `pip install torch`. A GPU is optional — the models are small enough for a laptop CPU. On Google Colab, upload the project folder and run the same cells.

In [ ]:
%cd ../..
import pandas as pd
from IPython.display import Image, display
from src.deep_learning import data as D, config as C

splits, scaler, features = D.prepare()
print("features:", len(features))
for name, s in splits.items():
    print(f"{name:5s} windows={s.X.shape}  period {str(s.timestamp.min())[:10]} -> {str(s.timestamp.max())[:10]}")

## 1. Baselines (no deep learning)

In [ ]:
!python -m src.deep_learning.baselines

## 2. Smoke test (≈1 min) — checks that PyTorch and the code work on your machine

In [ ]:
!python -m src.deep_learning.train --model transformer --smoke

## 3. Train the deep-learning models and the transformer
`--model all` trains mlp, lstm, gru and transformer one after another (early stopping on validation MAE). Use `--model transformer` to train just one, and e.g. `--epochs 60 --d-model 128 --layers 3` to tune it.

In [ ]:
!python -m src.deep_learning.train --model all --epochs 40

## 4. Compare everything

In [ ]:
!python -m src.deep_learning.report
display(pd.read_csv(C.REPORT_DIR / "model_comparison.csv").round(4))

In [ ]:
FIG = C.PROJECT_ROOT / "reports" / "figures" / "deep_learning"
for f in ["model_comparison.png", "forecast_vs_actual.png", "training_curves.png"]:
    if (FIG / f).exists():
        display(Image(filename=str(FIG / f)))

## How to read the results
- A neural network only counts as a win if it beats **both** `persistence` and `hist_gb_tabular` on the *test* columns (MAE / RMSE lower, R² and class accuracy higher).
- With ~46 k training windows and tabular sensor data, gradient boosting is often very hard to beat; if the transformer does not win, that is a legitimate finding — report it with the numbers.
- `class within 1 level` shows how often a wrong class is only one step away (e.g. High predicted as Severe), which matters more for a traffic dashboard than raw accuracy.
- Report the test numbers once, after you have finished choosing hyper-parameters on the validation split.